In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display


In [2]:
# Configure this cell for the run you want to inspect.
MC_NAME = "STRING340MC"
GEOMETRY = "102_string"
SCRATCH_BASE = Path("/scratch/kbas")

FLAVORS = ["Muon", "Electron", "Tau", "NC"]

MC_FOLDER = {
    "SPRING2026MC": "Spring2026MC",
    "STRING340MC": "String340MC",
}

def geometry_folder(mc_name: str, geometry: str) -> str:
    if mc_name == "STRING340MC" and geometry != "full_geometry":
        return geometry
    return "_".join(part.capitalize() for part in geometry.split("_"))

MC_DIR = SCRATCH_BASE / MC_FOLDER[MC_NAME]
GEO_DIR = geometry_folder(MC_NAME, GEOMETRY)

print(f"MC_DIR : {MC_DIR}")
print(f"GEO_DIR: {GEO_DIR}")


MC_DIR : /scratch/kbas/String340MC
GEO_DIR: 102_string


In [3]:
STATUS_RE = re.compile(r"===\s+(SUCCESS|FAILED)\s+elapsed=([0-9.]+)s\s+===")
FRAMES_RE = re.compile(
    r"frames_to_writer(?:_before_failure)?\s*:\s*DAQ=(\d+)\s+Simulation=(\d+)"
)
HEADER_RE = re.compile(r"^(infile|outfile|logfile|flavor|geometry|mc|gcd)\s*:\s*(.*)$", re.MULTILINE)
ERROR_RE = re.compile(r"^ERROR:\s*(.*)$", re.MULTILINE)


def file_size_mb(path: Path):
    if not path.exists():
        return None
    return path.stat().st_size / 1024 / 1024


def parse_pmt_log(log_path: Path, output_dir: Path) -> dict:
    text = log_path.read_text(errors="replace")

    headers = dict(HEADER_RE.findall(text))
    status_matches = STATUS_RE.findall(text)
    frame_matches = FRAMES_RE.findall(text)
    error_match = ERROR_RE.search(text)

    status = "unknown"
    elapsed_s = None
    if status_matches:
        status, elapsed = status_matches[-1]
        status = status.lower()
        elapsed_s = float(elapsed)

    daq_frames = None
    simulation_frames = None
    if frame_matches:
        daq_frames, simulation_frames = map(int, frame_matches[-1])

    output_path = Path(headers["outfile"]) if headers.get("outfile") else output_dir / f"{log_path.stem}.i3.gz"

    # Some runs/logs use /home/kbas/scratch while the login shell often shows /scratch/kbas.
    # If the logged output path is absent, also check the configured output directory.
    configured_output_path = output_dir / output_path.name
    output_exists = output_path.exists() or configured_output_path.exists()
    output_size_mb = file_size_mb(output_path)
    if output_size_mb is None:
        output_size_mb = file_size_mb(configured_output_path)

    return {
        "file_stem": log_path.stem,
        "input_file": Path(headers["infile"]).name if headers.get("infile") else None,
        "status": status,
        "daq_frames_to_writer": daq_frames,
        "simulation_frames_to_writer": simulation_frames,
        "elapsed_s": elapsed_s,
        "error": error_match.group(1) if error_match else None,
        "has_traceback": "Traceback (most recent call last):" in text,
        "log_path": str(log_path),
        "log_size_mb": file_size_mb(log_path),
        "output_path": str(output_path),
        "output_exists": output_exists,
        "output_size_mb": output_size_mb,
    }


def build_flavor_dataframe(flavor: str) -> pd.DataFrame:
    log_dir = MC_DIR / "Logs" / f"{flavor}_pmt_response_{GEO_DIR}"
    output_dir = MC_DIR / GEO_DIR / f"{flavor}_PMT_Response"

    rows = []
    for log_path in sorted(log_dir.glob("*.out")):
        row = parse_pmt_log(log_path, output_dir)
        row["flavor"] = flavor
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=[
            "flavor", "file_stem", "input_file", "status", "daq_frames_to_writer",
            "simulation_frames_to_writer", "elapsed_s", "error", "has_traceback",
            "output_exists", "output_size_mb", "log_size_mb", "log_path", "output_path",
        ])

    cols = [
        "flavor", "file_stem", "input_file", "status", "daq_frames_to_writer",
        "simulation_frames_to_writer", "elapsed_s", "error", "has_traceback",
        "output_exists", "output_size_mb", "log_size_mb", "log_path", "output_path",
    ]
    return df[cols].sort_values(["status", "daq_frames_to_writer", "file_stem"], ascending=[True, True, True])


In [4]:
dataframes = {flavor: build_flavor_dataframe(flavor) for flavor in FLAVORS}

Muon_df = dataframes["Muon"]
Electron_df = dataframes["Electron"]
Tau_df = dataframes["Tau"]
NC_df = dataframes["NC"]
all_df = pd.concat(dataframes.values(), ignore_index=True) if dataframes else pd.DataFrame()

for flavor, df in dataframes.items():
    print(f"\n=== {flavor} ===")
    print(f"logs parsed: {len(df)}")
    display(df.head(10))



=== Muon ===
logs parsed: 889


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
377,Muon,muon_muon_cls_126,muon_cls_126.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001107,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
488,Muon,muon_muon_cls_1362,muon_cls_1362.i3.gz,success,0.0,1.0,2.0,None,False,True,0.000111,0.001111,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
137,Muon,muon_muon_cls_1036,muon_cls_1036.i3.gz,success,2.0,1.0,3.2,None,False,True,0.616045,0.001111,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
472,Muon,muon_muon_cls_1347,muon_cls_1347.i3.gz,success,2.0,1.0,6.2,None,False,True,4.572164,0.001111,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
359,Muon,muon_muon_cls_1243,muon_cls_1243.i3.gz,success,3.0,1.0,263.5,None,False,True,393.827300,0.001113,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
91,Muon,muon_muon_cls_092,muon_cls_092.i3.gz,success,4.0,1.0,8.4,None,False,True,0.870749,0.001107,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
83,Muon,muon_muon_cls_084,muon_cls_084.i3.gz,success,7.0,1.0,9.3,None,False,True,3.317974,0.001107,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
797,Muon,muon_muon_cls_1644,muon_cls_1644.i3.gz,success,9.0,1.0,21.7,None,False,True,18.921356,0.001112,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
326,Muon,muon_muon_cls_1211,muon_cls_1211.i3.gz,success,10.0,1.0,28.5,None,False,True,25.971976,0.001113,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
463,Muon,muon_muon_cls_1339,muon_cls_1339.i3.gz,success,11.0,1.0,20.4,None,False,True,6.378261,0.001113,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...



=== Electron ===
logs parsed: 1804


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
188,Electron,electron_electron_gen_1080,electron_gen_1080.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
805,Electron,electron_electron_gen_1641,electron_gen_1641.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
855,Electron,electron_electron_gen_1687,electron_gen_1687.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
363,Electron,electron_electron_gen_124,electron_gen_124.i3.gz,success,2.0,1.0,5.9,None,False,True,0.761574,0.001153,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
641,Electron,electron_electron_gen_1492,electron_gen_1492.i3.gz,success,2.0,1.0,4.3,None,False,True,0.851379,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
677,Electron,electron_electron_gen_1525,electron_gen_1525.i3.gz,success,2.0,1.0,20.2,None,False,True,33.996052,0.001158,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
914,Electron,electron_electron_gen_1740,electron_gen_1740.i3.gz,success,2.0,1.0,6.6,None,False,True,0.262742,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1092,Electron,electron_electron_gen_1902,electron_gen_1902.i3.gz,success,2.0,1.0,4.9,None,False,True,0.283454,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1255,Electron,electron_electron_gen_2050,electron_gen_2050.i3.gz,success,2.0,1.0,6.8,None,False,True,0.276377,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1644,Electron,electron_electron_gen_2404,electron_gen_2404.i3.gz,success,2.0,1.0,4.8,None,False,True,0.326097,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...



=== Tau ===
logs parsed: 1920


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
432,Tau,tau_tau_gen_1301,tau_gen_1301.i3.gz,success,1.0,1.0,6.0,None,False,True,0.111687,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
1735,Tau,tau_tau_gen_2486,tau_gen_2486.i3.gz,success,1.0,1.0,8.0,None,False,True,0.115835,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
178,Tau,tau_tau_gen_1070,tau_gen_1070.i3.gz,success,2.0,1.0,7.4,None,False,True,0.291678,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
549,Tau,tau_tau_gen_1408,tau_gen_1408.i3.gz,success,2.0,1.0,7.7,None,False,True,1.538775,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
1074,Tau,tau_tau_gen_1885,tau_gen_1885.i3.gz,success,2.0,1.0,5.4,None,False,True,0.398766,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
1481,Tau,tau_tau_gen_2255,tau_gen_2255.i3.gz,success,2.0,1.0,8.1,None,False,True,0.345470,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
1568,Tau,tau_tau_gen_2334,tau_gen_2334.i3.gz,success,2.0,1.0,9.4,None,False,True,0.276174,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
2,Tau,tau_tau_gen_002,tau_gen_002.i3.gz,success,3.0,1.0,10.9,None,False,True,0.542375,0.001097,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
91,Tau,tau_tau_gen_091,tau_gen_091.i3.gz,success,3.0,1.0,5.4,None,False,True,0.647402,0.001096,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...
239,Tau,tau_tau_gen_1126,tau_gen_1126.i3.gz,success,3.0,1.0,9.9,None,False,True,1.538826,0.001100,/scratch/kbas/String340MC/Logs/Tau_pmt_respons...,/scratch/kbas/String340MC/102_string/Tau_PMT_R...



=== NC ===
logs parsed: 2352


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
1569,NC,nc_nc_gen_2335,nc_gen_2335.i3.gz,success,0.0,0.0,1.5,None,False,True,0.000019,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
2199,NC,nc_nc_gen_2908,nc_gen_2908.i3.gz,success,0.0,1.0,4.6,None,False,True,0.000111,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
224,NC,nc_nc_gen_1112,nc_gen_1112.i3.gz,success,1.0,1.0,5.2,None,False,True,0.115081,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
376,NC,nc_nc_gen_1250,nc_gen_1250.i3.gz,success,1.0,1.0,4.5,None,False,True,0.117740,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
545,NC,nc_nc_gen_1404,nc_gen_1404.i3.gz,success,1.0,1.0,6.7,None,False,True,0.668936,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
643,NC,nc_nc_gen_1493,nc_gen_1493.i3.gz,success,1.0,1.0,9.2,None,False,True,1.196200,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
749,NC,nc_nc_gen_159,nc_gen_159.i3.gz,success,1.0,1.0,4.7,None,False,True,0.123994,0.001084,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
965,NC,nc_nc_gen_1786,nc_gen_1786.i3.gz,success,1.0,1.0,6.6,None,False,True,0.124555,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
978,NC,nc_nc_gen_1798,nc_gen_1798.i3.gz,success,1.0,1.0,9.8,None,False,True,0.130630,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...
1098,NC,nc_nc_gen_1907,nc_gen_1907.i3.gz,success,1.0,1.0,4.8,None,False,True,0.138736,0.001088,/scratch/kbas/String340MC/Logs/NC_pmt_response...,/scratch/kbas/String340MC/102_string/NC_PMT_Re...


In [5]:
summary_rows = []
for flavor, df in dataframes.items():
    counts = df["status"].value_counts(dropna=False).to_dict() if not df.empty else {}
    summary_rows.append({
        "flavor": flavor,
        "n_logs": len(df),
        "success": counts.get("success", 0),
        "failed": counts.get("failed", 0),
        "unknown": counts.get("unknown", 0),
        "total_daq_frames_to_writer": df["daq_frames_to_writer"].sum() if not df.empty else 0,
        "median_daq_frames_to_writer": df["daq_frames_to_writer"].median() if not df.empty else None,
        "zero_daq_logs": int((df["daq_frames_to_writer"].fillna(0) == 0).sum()) if not df.empty else 0,
        "missing_outputs": int((~df["output_exists"].fillna(False)).sum()) if not df.empty else 0,
    })

summary = pd.DataFrame(summary_rows)
display(summary)


,flavor,n_logs,success,failed,unknown,total_daq_frames_to_writer,median_daq_frames_to_writer,zero_daq_logs,missing_outputs
0,Muon,889,878,0,11,19007.0,22.0,13,0
1,Electron,1804,1800,0,4,17506.0,10.0,7,0
2,Tau,1920,1916,0,4,19074.0,10.0,4,0
3,NC,2352,2347,0,5,16138.0,7.0,7,0


In [6]:
problem_rows = []
for flavor, df in dataframes.items():
    if df.empty:
        continue
    problem = df[
        (df["status"] != "success")
        | (df["daq_frames_to_writer"].isna())
        | (df["daq_frames_to_writer"].fillna(0) == 0)
        | (~df["output_exists"].fillna(False))
    ].copy()
    problem_rows.append(problem)

problems = pd.concat(problem_rows, ignore_index=True) if problem_rows else pd.DataFrame()
display(problems)


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
0,Muon,muon_muon_cls_126,muon_cls_126.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001107,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
1,Muon,muon_muon_cls_1362,muon_cls_1362.i3.gz,success,0.0,1.0,2.0,None,False,True,0.000111,0.001111,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
2,Muon,muon_muon_cls_1610,muon_cls_1610.i3.gz,unknown,NaN,NaN,NaN,None,False,True,415.656250,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
3,Muon,muon_muon_cls_1697,muon_cls_1697.i3.gz,unknown,NaN,NaN,NaN,None,False,True,192.937500,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
4,Muon,muon_muon_cls_1710,muon_cls_1710.i3.gz,unknown,NaN,NaN,NaN,None,False,True,4.945312,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
5,Muon,muon_muon_cls_1717,muon_cls_1717.i3.gz,unknown,NaN,NaN,NaN,None,False,True,8.750000,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
6,Muon,muon_muon_cls_172,muon_cls_172.i3.gz,unknown,NaN,NaN,NaN,None,False,True,17.062500,0.000492,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
7,Muon,muon_muon_cls_1722,muon_cls_1722.i3.gz,unknown,NaN,NaN,NaN,None,False,True,12.210938,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
8,Muon,muon_muon_cls_1723,muon_cls_1723.i3.gz,unknown,NaN,NaN,NaN,None,False,True,9.132812,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...
9,Muon,muon_muon_cls_1724,muon_cls_1724.i3.gz,unknown,NaN,NaN,NaN,None,False,True,5.203125,0.000495,/scratch/kbas/String340MC/Logs/Muon_pmt_respon...,/scratch/kbas/String340MC/102_string/Muon_PMT_...


In [7]:
# Convenience: inspect one flavor interactively.
flavor = "Electron"
df = dataframes[flavor]

display(df.sort_values("daq_frames_to_writer", ascending=True).head(20))
display(df.sort_values("output_size_mb", ascending=True).head(20))


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
188,Electron,electron_electron_gen_1080,electron_gen_1080.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
805,Electron,electron_electron_gen_1641,electron_gen_1641.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
855,Electron,electron_electron_gen_1687,electron_gen_1687.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1255,Electron,electron_electron_gen_2050,electron_gen_2050.i3.gz,success,2.0,1.0,6.8,None,False,True,0.276377,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1666,Electron,electron_electron_gen_2424,electron_gen_2424.i3.gz,success,2.0,1.0,11.9,None,False,True,0.377376,0.001158,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1680,Electron,electron_electron_gen_2437,electron_gen_2437.i3.gz,success,2.0,1.0,10.5,None,False,True,0.923429,0.001158,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1644,Electron,electron_electron_gen_2404,electron_gen_2404.i3.gz,success,2.0,1.0,4.8,None,False,True,0.326097,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
363,Electron,electron_electron_gen_124,electron_gen_124.i3.gz,success,2.0,1.0,5.9,None,False,True,0.761574,0.001153,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
914,Electron,electron_electron_gen_1740,electron_gen_1740.i3.gz,success,2.0,1.0,6.6,None,False,True,0.262742,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
677,Electron,electron_electron_gen_1525,electron_gen_1525.i3.gz,success,2.0,1.0,20.2,None,False,True,33.996052,0.001158,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
188,Electron,electron_electron_gen_1080,electron_gen_1080.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
805,Electron,electron_electron_gen_1641,electron_gen_1641.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
855,Electron,electron_electron_gen_1687,electron_gen_1687.i3.gz,success,0.0,0.0,1.2,None,False,True,0.000019,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
914,Electron,electron_electron_gen_1740,electron_gen_1740.i3.gz,success,2.0,1.0,6.6,None,False,True,0.262742,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1255,Electron,electron_electron_gen_2050,electron_gen_2050.i3.gz,success,2.0,1.0,6.8,None,False,True,0.276377,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1714,Electron,electron_electron_gen_2468,electron_gen_2468.i3.gz,unknown,NaN,NaN,NaN,None,False,True,0.281250,0.000529,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1092,Electron,electron_electron_gen_1902,electron_gen_1902.i3.gz,success,2.0,1.0,4.9,None,False,True,0.283454,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1644,Electron,electron_electron_gen_2404,electron_gen_2404.i3.gz,success,2.0,1.0,4.8,None,False,True,0.326097,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1666,Electron,electron_electron_gen_2424,electron_gen_2424.i3.gz,success,2.0,1.0,11.9,None,False,True,0.377376,0.001158,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...
1169,Electron,electron_electron_gen_1972,electron_gen_1972.i3.gz,success,3.0,1.0,8.4,None,False,True,0.384189,0.001157,/scratch/kbas/String340MC/Logs/Electron_pmt_re...,/scratch/kbas/String340MC/102_string/Electron_...


In [8]:
# For each flavor: among failed files, how many reached 0 DAQ frames vs at least some DAQ frames?
failed_daq_rows = []

for flavor, df in dataframes.items():
    failed = df[df["status"] == "failed"].copy()
    failed_daq = failed["daq_frames_to_writer"]

    n_failed = len(failed)
    n_failed_zero_daq = int((failed_daq.fillna(0) == 0).sum())
    n_failed_positive_daq = int((failed_daq.fillna(0) > 0).sum())
    n_failed_missing_daq = int(failed_daq.isna().sum())

    failed_daq_rows.append({
        "flavor": flavor,
        "failed_total": n_failed,
        "failed_zero_daq": n_failed_zero_daq,
        "failed_positive_daq": n_failed_positive_daq,
        "failed_missing_daq": n_failed_missing_daq,
        "failed_positive_daq_fraction": n_failed_positive_daq / n_failed if n_failed else None,
        "failed_zero_daq_fraction": n_failed_zero_daq / n_failed if n_failed else None,
        "failed_daq_min": failed_daq.min() if n_failed else None,
        "failed_daq_median": failed_daq.median() if n_failed else None,
        "failed_daq_max": failed_daq.max() if n_failed else None,
    })

failed_daq_summary = pd.DataFrame(failed_daq_rows)
display(failed_daq_summary)

# Detailed list of failed files with 0 or missing DAQ frames.
failed_zero_daq_files = all_df[
    (all_df["status"] == "failed")
    & (all_df["daq_frames_to_writer"].fillna(0) == 0)
].sort_values(["flavor", "file_stem"])

display(failed_zero_daq_files)


,flavor,failed_total,failed_zero_daq,failed_positive_daq,failed_missing_daq,failed_positive_daq_fraction,failed_zero_daq_fraction,failed_daq_min,failed_daq_median,failed_daq_max
0,Muon,0,0,0,0,None,None,None,None,None
1,Electron,0,0,0,0,None,None,None,None,None
2,Tau,0,0,0,0,None,None,None,None,None
3,NC,0,0,0,0,None,None,None,None,None


,flavor,file_stem,input_file,status,daq_frames_to_writer,simulation_frames_to_writer,elapsed_s,error,has_traceback,output_exists,output_size_mb,log_size_mb,log_path,output_path
